# B.1: SHAP-IMV with simulated data

Simulate 1,000 binary outcomes from two predictors and use `BinaryIMV` to evaluate every feature subset with logistic regression in three stratified folds. Each subset is compared with a constant-only model fitted on the training observations.

Run the cell below from any working directory. The base `imvpy` installation supplies the required NumPy, pandas, and scikit-learn dependencies; no data download is needed.

In [1]:
# Install once: python -m pip install imvpy
import numpy as np
import pandas as pd
from imvpy import BinaryIMV
from sklearn.linear_model import LogisticRegression

# 1. Simulate a binary outcome from two predictors.
rng = np.random.default_rng(42)
x = rng.normal(size=(1000, 2))
p = 1 / (1 + np.exp(-(2 * x[:, 0] + x[:, 1] - 1)))
data = pd.DataFrame(x, columns=["x1", "x2"])
data["y"] = rng.binomial(1, p)

# 2. Evaluate every feature subset on held-out folds.
evaluator = BinaryIMV(
    data=data,
    outcome_variable="y",
    optional_explanatory_variables=["x1", "x2"],
    model_creator=lambda: LogisticRegression(),
    split_method="stratified_kfold",
    n_splits=3,
    random_seed=42,
)
evaluator.run_evaluation()

# 3. Attribute the predictive gain to each feature.
for feature in ["x1", "x2"]:
    value = evaluator.calculate_imvshapley_value(feature)
    print(f"SHAP-IMV for {feature}: {value:.3f}")

  0%|          | 0/4 [00:00<?, ?it/s]

SHAP-IMV for x1: 0.209
SHAP-IMV for x2: 0.056


The global SHAP-IMV values allocate predictive gain across features: this example returns 0.209 for `x1` and 0.056 for `x2`. These are global model-fit attributions, rather than explanations of individual predictions.